# 第35章　データの読み込みを実装する：DICOM・NIfTI・3次元画像

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 35.1　対応表から画像とラベルを読み込む

In [ ]:
import torch, pydicom, numpy as np
from torch.utils.data import Dataset, DataLoader

# モダリティ別の前処理の章（29.1）と同じ関数（ct_window と load_dicom_hu）。章単位で動かすため再掲
def ct_window(hu, center, width):
    low, high = center - width/2, center + width/2
    return (np.clip(hu, low, high) - low) / (high - low)   # 0〜1へ

def load_dicom_hu(path):
    """CT専用。確認できない入力は処理を進めず、例外で止める（29.1と同じ）。"""
    ds = pydicom.dcmread(path)
    if getattr(ds, "Modality", "") != "CT":
        raise ValueError("CT以外のモダリティ。CT値は定義されない")
    if int(getattr(ds, "NumberOfFrames", 1)) != 1:
        raise ValueError("マルチフレーム。別の変換経路を確認してから扱う")
    if "RescaleSlope" not in ds or "RescaleIntercept" not in ds:
        raise ValueError("CT値への変換係数がない。既定値で埋めず、変換不能として扱う")
    rescale_type = str(getattr(ds, "RescaleType", "")).strip().upper()
    image_type = list(getattr(ds, "ImageType", []))
    multi_energy = str(ds[(0x0018, 0x9361)].value).strip() if (0x0018, 0x9361) in ds else "NO"
    hu_by_image_type = (len(image_type) >= 3 and image_type[0] == "ORIGINAL"
                        and image_type[2] != "LOCALIZER" and multi_energy == "NO")
    if rescale_type != "HU" and not (not rescale_type and hu_by_image_type):
        raise ValueError("変換後の単位がHUと確認できない。別の変換経路で扱う")
    arr = ds.pixel_array.astype(np.float32)
    hu = arr * float(ds.RescaleSlope) + float(ds.RescaleIntercept)   # 格納値→CT値
    return hu, ds

class CTDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)     # 対応表（case_id, image_path, label）
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        hu, ds = load_dicom_hu(row["image_path"])       # 入力形式とHU単位を確認する（29.1と同じ関数）
        img = ct_window(hu, center=40, width=400)        # 前処理（モダリティ別の前処理の章）
        img = torch.from_numpy(img).float().unsqueeze(0) # (1, H, W)
        label = int(row["label"])
        if self.transform:
            img = self.transform(img)
        return img, label

## 35.2　3次元の画像（NIfTI）を読み込む

In [ ]:
# 読み込みもMONAIのCompose（LoadImaged）が担当するので、ここではnibabelを直接呼ばない
class VolumeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df, self.transform = df.reset_index(drop=True), transform
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = {"image": row["image_path"], "label": row["label_path"]}
        if self.transform:
            data = self.transform(data)          # MONAIのCompose（読み込み含む）
        return data["image"], data["label"]
    def __len__(self):
        return len(self.df)
# この例の transform は「単一の辞書を返す」ものに限る。第27章の RandCropByPosNegLabeld（num_samples>1）の
# ように辞書のリストを返す変換を使うなら、リストのまま返して MONAI の DataLoader（list_data_collate）で束ねる

## トラブルシュート：DataLoaderの停止と、結果が毎回変わる問題

In [ ]:
import numpy as np

def seed_everything(seed=42):
    import random, os
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True    # 畳み込みを決定的アルゴリズムに
    torch.backends.cudnn.benchmark = False        # 実行時の計測によるアルゴリズム選択を無効にする

def worker_init_fn(worker_id):                     # 各ワーカーにも異なるが再現可能なシードを
    np.random.seed((torch.initial_seed() + worker_id) % 2**32)   # 先に足してから 2**32 の剰余を取る（NumPy のシードの上限を超えないように）

loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                    num_workers=4, worker_init_fn=worker_init_fn)

## collate_fnを書く：可変サイズ・メタデータ・読込エラーのあるサンプルに対応する

In [ ]:
from torch.utils.data import default_collate

def safe_collate(batch):
    kept, failed = [], []
    for b in batch:
        if isinstance(b, dict) and b.get("error"):   # Dataset側が返した失敗レコード
            failed.append((b["case_id"], b["error"]))
            continue
        kept.append(b)
    # 成功分と失敗分を「組」にして返す。collate_fn は num_workers>0 のとき worker プロセスの中で
    # 動くので、グローバルのリストに追記しても親プロセスには届かない。戻り値で返す。
    return {"data": default_collate(kept) if kept else None, "failed": failed}

# Dataset側: __getitem__ を try/except で包み、失敗時は {"case_id":..., "error": str(e)} を返す
# 学習ループ側（親プロセス）:
#   FAILED = []                               # 除外した症例の情報を必ず記録する。件数と理由が分からない除外では性能評価に偏りが生じるおそれがある
#   for batch in loader:
#       FAILED.extend(batch["failed"])        # worker から戻ってきた失敗分を回収する
#       if batch["data"] is None: continue    # 全サンプルが読み込みに失敗したバッチはスキップ（件数は FAILED に残る）
#       imgs, labels = batch["data"]
# 学習の最後に: 全対象 / 成功 / 失敗 を突き合わせ、FAILED を必ず出力する。
# 評価では、失敗件数を「評価不能」として報告し、分母から除外した場合はその件数も明記する
loader = DataLoader(ds, batch_size=16, collate_fn=safe_collate, num_workers=4)

In [ ]:
import torch, torch.nn.functional as F

def pad_collate(batch):
    imgs, labels = zip(*batch)                       # 各 img は (C, H, W)、大きさは画像ごとに異なる
    H = max(i.shape[-2] for i in imgs)
    W = max(i.shape[-1] for i in imgs)
    padded = [F.pad(i, (0, W - i.shape[-1], 0, H - i.shape[-2])) for i in imgs]  # 右・下に0詰め
    return torch.stack(padded), torch.tensor(labels)

In [ ]:
def meta_collate(batch):
    imgs   = torch.stack([b["image"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    meta   = [b["meta"] for b in batch]              # {"case_id":..., "spacing":...} のリスト
    return {"image": imgs, "label": labels, "meta": meta}

## 35.4　拡張は、学習と検証で分ける

In [ ]:
from monai import transforms as T

# 35.2 で述べたとおり「単一の辞書を返す」変換だけで組む。第27章の train_tf は
# RandCropByPosNegLabeld で辞書のリストを返すので、そのまま渡すと data["image"] で TypeError になる
vol_val_tf = T.Compose([
    T.LoadImaged(keys=["image", "label"]),
    T.EnsureChannelFirstd(keys=["image", "label"]),
])
vol_train_tf = T.Compose([
    T.LoadImaged(keys=["image", "label"]),
    T.EnsureChannelFirstd(keys=["image", "label"]),
    T.RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),   # 強度の拡張は画像だけ
])
train_ds = VolumeDataset(train_df, transform=vol_train_tf)   # 拡張あり
val_ds   = VolumeDataset(val_df,   transform=vol_val_tf)     # 拡張なし